# 3 · Train and predict

Train on the image/label pairs saved in notebook 2, then predict land cover for the KHDTK forest from notebook 2.

The flow is: **load pairs → prepare → cut chips → train → predict chips → save maps → save the model**.

**Data and interpretation.** These are expert-annotated training labels from [Tait et al. (2021)](https://doi.org/10.1594/PANGAEA.933475), paired with Sentinel-2 L2A imagery in the [teaching subset](https://huggingface.co/datasets/fatmur/dynamicworld-300). They have ten classes, including cloud, and nodata 255. They are not the automated nine-class Dynamic World probability product.

Keep tiles in their original train/validation/test splits before making chips. Nearby or overlapping tiles can still leak spatial information: inspect footprints before claiming geographic generalization. These short runs are workflow checks, not evidence of mapping accuracy.

**Run:** use the repository `.venv` kernel with `examples/` as the working directory. Requires `uv sync` and notebook 2's saved files.

In [ ]:
import warnings
from pathlib import Path

import kornia.augmentation as K
import lightning as L
import torch
import xarray as xr
import yaml
from torch.utils.data import DataLoader, Dataset

import geosave as gs

L.seed_everything(0)
warnings.filterwarnings("ignore", ".*does not have many workers.*")

## Load the classes

Use the dataset's class names, colours and ignore index throughout this notebook.

In [ ]:
classes = yaml.safe_load(Path("data/dynamic_world/classes.yaml").read_text())
IGNORE_INDEX = classes["ignore_index"]
NUM_CLASSES = len(classes["class_map"])
classes["class_map"]

## Load the saved pairs

Each catalog row is one image/label pair. Select a split with pandas and load each row with `.gs.to_dataset()`: image bands and label on that pair's grid. Pairs sit in different UTM zones, so load them one at a time.

In [ ]:
catalog = gs.open_vector("data/my_dataset/catalog.parquet")


def load_pairs(rows):
    pairs = []
    for index in range(len(rows)):
        pair = rows.iloc[[index]].gs.to_dataset().squeeze("time")
        pairs.append(pair.load())
    return pairs


train_pairs = load_pairs(catalog[catalog["split"] == "train"])
val_pairs = load_pairs(catalog[catalog["split"] == "val"])
test_pairs = load_pairs(catalog[catalog["split"] == "test"])
train_pairs[0]

In [ ]:
pair = train_pairs[0]
label = pair["label"].gs.write_flags(classes["class_map"], classes["color_map"])

pair.gs.plot(rgb=("B04", "B03", "B02"), xlabel="") + label.gs.plot(xlabel="")

## Prepare the acquired imagery

Unlike the complete teaching tiles used in notebook 0, these newly acquired scenes can have missing coverage. Decode reflectance. Where an image has no observation, use zero as its input and 255 as its target so the loss ignores that location. Label IDs are otherwise unchanged.

The prepared pair is a DataTree with an `image` and a `label` node on one grid: the shape `gs.Chipper` cuts.

In [ ]:
BANDS = ["B02", "B03", "B04", "B08"]


def prepare_pair(pair):
    reflectance = pair[BANDS].gs.mask_and_scale()
    observed = reflectance.to_array().notnull().all("variable")

    image = reflectance.fillna(0)
    label = pair[["label"]].where(observed, IGNORE_INDEX)
    return xr.DataTree.from_dict({"image": image, "label": label})


train_pairs = [prepare_pair(pair) for pair in train_pairs]
val_pairs = [prepare_pair(pair) for pair in val_pairs]
test_pairs = [prepare_pair(pair) for pair in test_pairs]

## Cut matching image and label chips

The model takes 128 × 128 pixels. `gs.Chipper` lays chips over every pair and cuts image and label together when indexed; construction reads only the grids. `mode="shift"` moves the last chip of each row inward so it ends at the scene's edge: no pixels are invented, so neither image nor label needs padding. Those edge chips overlap their neighbours, so a thin strip of evaluation pixels is counted twice.

In [ ]:
train_chips = gs.Chipper(train_pairs, size=128, mode="shift")
val_chips = gs.Chipper(val_pairs, size=128, mode="shift")
test_chips = gs.Chipper(test_pairs, size=128, mode="shift")

len(train_chips), len(val_chips), len(test_chips)

## Return tensors for training

The Dataset only selects a chip and converts its image and label to tensors.

In [ ]:
class DynamicWorld(Dataset):
    def __init__(self, chips):
        self.chips = chips

    def __len__(self):
        return len(self.chips)

    def __getitem__(self, index):
        chip = self.chips[index]
        image = chip["image"].to_dataset().gs.to_tensor(dtype="float32")
        label = chip["label"].to_dataset()["label"].gs.to_tensor(dtype="int64")

        return image, label


train_loader = DataLoader(DynamicWorld(train_chips), batch_size=8, shuffle=True)
val_loader = DataLoader(DynamicWorld(val_chips), batch_size=8)
test_loader = DataLoader(DynamicWorld(test_chips), batch_size=8)

## Build the model

Keep the Lightning training loop from notebook 1. The model is built from registered parts by name, which lets us save it as a release at the end. Apply the same flips to each image and label during training. Validation and test use the unaugmented chips and ignore unlabelled pixels.

In [ ]:
from torch import nn
from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassJaccardIndex

from geosave.model import build_model


class Segmenter(L.LightningModule):
    def __init__(self, num_classes, ignore_index, in_channels=4, input_size=128, width=16):
        super().__init__()

        self.model = build_model(
            {
                "encoder": {
                    "name": "small_cnn",
                    "init_args": {
                        "in_channels": in_channels,
                        "input_size": input_size,
                        "width": width,
                    },
                },
                "decoder": {"name": "unet"},
                "head": {"name": "segmentation", "init_args": {"num_classes": num_classes}},
            }
        )

        self.augment = K.AugmentationSequential(
            K.RandomHorizontalFlip(),
            K.RandomVerticalFlip(),
            data_keys=["image", "mask"],
            keepdim=True,
        )

        self.val_metrics = MetricCollection(
            {
                "miou": MulticlassJaccardIndex(
                    num_classes=num_classes, ignore_index=ignore_index
                ),
                "accuracy": MulticlassAccuracy(
                    num_classes=num_classes, ignore_index=ignore_index, average="micro"
                ),
            },
            prefix="val_",
        )
        self.test_metrics = self.val_metrics.clone(prefix="test_")

        self.criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)

    def forward(self, image):
        return self.model(image=image)

    def training_step(self, batch, batch_index):
        image, label = self.augment(*batch)
        logits = self(image)
        loss = self.criterion(logits, label)

        self.log("train_loss", loss)

        return loss

    def validation_step(self, batch, batch_index):
        image, label = batch
        logits = self(image)

        self.val_metrics(logits, label)
        self.log_dict(self.val_metrics)

    def test_step(self, batch, batch_index):
        image, label = batch
        logits = self(image)

        self.test_metrics(logits, label)
        self.log_dict(self.test_metrics)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-2)


module = Segmenter(num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX)
module

## Train and evaluate

Run six short epochs, using six training batches per epoch. Validation covers all chips from the two validation scenes. Logs and checkpoints go to `lightning_logs/03_train/`; rerunning starts a fresh model.

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

logger = TensorBoardLogger("lightning_logs", name="03_train", version=0)
checkpoint = ModelCheckpoint(save_last=True)

trainer = L.Trainer(
    max_epochs=6,
    limit_train_batches=6,
    log_every_n_steps=1,
    logger=logger,
    callbacks=[checkpoint],
)
trainer.fit(module, train_loader, val_loader)

Evaluate on the held-out test split.

In [ ]:
trainer.test(module, test_loader)

### Optional: inspect training logs

Run this in a separate cell when you want TensorBoard. It starts a local service.

```python
%load_ext tensorboard
%tensorboard --logdir lightning_logs
```

## Open the forest observations

The forest has no reference labels. Notebook 2's catalog loads as one Dataset: six dates on the forest grid. Prepare it as the training images were, and keep the observation mask for the final maps.

In [ ]:
boundary = gs.open_vector("khdtk.geojson")
forest = gs.open_vector("data/khdtk/catalog.parquet").gs.to_dataset().load()

reflectance = forest[BANDS].gs.mask_and_scale()
observed = reflectance.to_array().notnull().all("variable")
image = reflectance.fillna(0)
image

## Predict overlapping chips

Give each date its own tree and cut chips that overlap by half (`stride=64`). `gs.ChipMerger` knows where each chip came from, so a prediction needs only its chip index. Its `hamming` window weights chip centres over their rims when overlaps are blended.

In [ ]:
trees = []
for position in range(image.sizes["time"]):
    trees.append(xr.DataTree.from_dict({"image": image.isel(time=position)}))

chips = gs.Chipper(trees, size=128, stride=64)
merger = gs.ChipMerger(chips.layout, window="hamming")

module.eval()
with torch.inference_mode():
    for index in range(len(chips)):
        pixels = chips[index]["image"].to_dataset().gs.to_tensor(dtype="float32")
        scores = module(pixels.unsqueeze(0).to(module.device))[0].cpu().numpy()
        merger.add(index, scores)

## Put the scores back on the map

`merger.merge(position)` returns one date's class scores on the forest grid. Choose the highest-scoring class (class IDs are 0–9, the score positions), leave missing observations as 255, and attach the class names and colours.

In [ ]:
maps = []
for position in range(len(trees)):
    scores = merger.merge(position)
    maps.append(scores.argmax("band").astype("uint8"))

cover = xr.concat(maps, dim=image.time)
cover = cover.where(observed, IGNORE_INDEX).rename("label")
cover = cover.gs.write_nodata(IGNORE_INDEX)
cover = cover.gs.write_flags(classes["class_map"], classes["color_map"])
cover.gs.plot(xlabel="")

## Save the maps

Write one TIFF per date and a catalog pointing to those files. Keep the catalog and raster folder together when moving the results.

In [ ]:
folder = Path("data/khdtk_pred")

cover_catalog = cover.to_dataset().gs.to_catalog(
    folder, id_format="{start:%Y%m%d}", overwrite=True
)
cover_catalog.gs.to_geoparquet(folder / "catalog.parquet", overwrite=True)
cover_catalog[["id", "datetime"]]

In [ ]:
cover.to_dataset().gs.explore(basemap="Imagery") * boundary.gs.explore()

## Keep this trained model for the next observation

A Lightning checkpoint keeps the optimizer and training state for resuming a run. To hand the model on, save a release instead: `GeoSaveModel` stores the registered architecture and trained weights (`config.json`, `model.safetensors`). Keep the class definitions beside it.

Rerunning this cell replaces the release in `data/landcover/`. Keep a separate folder for each experiment you intend to compare.

In [ ]:
from geosave.model.release import GeoSaveModel

handoff = Path("data/landcover")

module.eval()
GeoSaveModel.from_chain(module.model).save_pretrained(handoff)
(handoff / "classes.yaml").write_text(yaml.safe_dump(classes, sort_keys=False))
sorted(path.name for path in handoff.iterdir())

## More things you can do with the forest maps (optional)

The maps and trained model are saved. Continue to [notebook 4](04_recipe.ipynb), or try these review tasks.

### Give the forest manager a class summary
`.gs.crop` keeps only predictions inside the project boundary. Show the modeled class distribution beside the map, and keep missing pixels out of the denominator.

In [ ]:
cover.gs.crop(boundary).gs.class_percent()

### Review suspected changes with the original imagery
A drop in predicted tree cover is a reason to inspect the two dates side by side. Clouds, shadows, season, and model error can all change the prediction. Confirm a suspected change with independent observations before calling it forest loss.

### Prepare a GIS review layer
Vectorize one dated class map with `.gs.vectorize(value_name="class_id")`, join the class names, and let reviewers annotate disputed areas. Preserve the date and model run with the layer.

**Next:** [reuse the saved model on another observation](04_recipe.ipynb) without changing band order, reflectance scaling, or missing-pixel handling.